# DBFS (Databricks File System) and Utilities
Learn how to access data from Azure Data Lake and use Databricks utilities.

### Access Data from Data Lake

In [0]:
# Pulling Azure service principal credentials from Databricks secret store
client_id     = dbutils.secrets.get(scope="my-scope", key="client-id")
client_secret = dbutils.secrets.get(scope="my-scope", key="client-secret")
tenant_id     = dbutils.secrets.get(scope="my-scope", key="tenant-id")

In [0]:
print(client_id)
print(client_secret)
print(tenant_id)
# Note: Actual values will show as [REDACTED] for security

In [0]:
storage_account = "storage4spotifyproject"

# Connecting to Azure Data Lake using OAuth with service principal
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

# Azure AD endpoint to get the access token
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

## Databricks Utilities (dbutils)
dbutils provides access to file system operations, widgets, and secrets.

In [0]:
display(dbutils.fs)

### ABFSS Protocol
abfss:// is the secure protocol to talk directly to Azure Data Lake without mounting.

In [0]:
dbutils.fs.ls(f"abfss://bronze@storage4spotifyproject.dfs.core.windows.net/")

In [0]:
df = spark.read.csv(f"abfss://bronze@storage4spotifyproject.dfs.core.windows.net/movies.csv", header=True, inferSchema=True)
df.display()

### dbutils.widgets
Used to create input parameters in notebooks — so instead of hardcoding values, you can pass them dynamically.

In [0]:
# Creates a dropdown to switch between environments
dbutils.widgets.dropdown("env", "dev", ["dev", "staging", "prod"])

In [0]:
env = dbutils.widgets.get("env")
print(env)

In [0]:
# Used in ETL jobs to process specific date ranges
dbutils.widgets.text("start_date", "2024-01-01")
dbutils.widgets.text("end_date",   "2024-12-31")

In [0]:
# Medallion architecture — pick which layer to process
dbutils.widgets.dropdown("layer", "bronze", ["bronze", "silver", "gold"])

In [0]:
# Run same notebook for different tables
dbutils.widgets.text("table_name", "sales")

In [0]:
# Full load vs incremental load
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])

### dbutils.secrets
Used to securely access credentials stored in Databricks Secret Store or Azure Key Vault.

In [0]:
# Lists all scopes — find your key vault scope name
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list(scope="myADB-scope")

In [0]:
dbutils.secrets.get(scope="myADB-scope", key="app-id-secret")